# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Unit of prediction: a tutoring session (row). Students appear in several rows and there is no date column, so the split is by **whole students** (checklist 1, 2).

Feature timing (columns.csv): `student_key` is an identifier (dropped). `enrol_year`, `school_code`, `prior_score` are known before the session. `session_minutes` and `hints_used` are recorded during the session; I assume the prediction is made at the end of the tutoring, just before the quiz, so they are allowed. A `logreg_pre_session` candidate (without them) covers the case where prediction must happen at session start.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, brier_score_loss
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.student_key.nunique(), 'students; pass rate', df.passed_quiz.mean().round(3))
print(df.isna().sum().sum(), 'missing values')

## 1. Hold out a test set of whole students, before any modelling

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr,te=next(gss.split(df,groups=df.student_key))
train,test=df.iloc[tr].reset_index(drop=True),df.iloc[te].reset_index(drop=True)
assert not set(train.student_key)&set(test.student_key)
print(len(train),len(test),'rows;',train.student_key.nunique(),test.student_key.nunique(),'students')
ALL=['enrol_year','school_code','prior_score','session_minutes','hints_used']  # student_key dropped
PRE=['enrol_year','school_code','prior_score']
y,g=train.passed_quiz,train.student_key

## 2. Candidates, declared before any scoring
1. **baseline**: constant training pass rate.
2. **logreg_all**: scaled logistic regression on all five features, C in {0.01, 0.1, 1, 10}.
3. **logreg_pre_session**: same, on the three pre-session features only.
4. **gbm_all**: `HistGradientBoostingClassifier`, max_depth in {2, 3}, learning rate 0.05.

`school_code` is used as a numeric column (a simplification). Selection metric: ROC AUC on out-of-fold predictions; the threshold is then chosen on those same CV predictions to maximise balanced accuracy. CV is 5-fold `GroupKFold` by student inside the training rows; imputation and scaling are inside the pipelines, so they are fit per fold.

In [ ]:
def mk(name,**p):
    if name=='baseline': return DummyClassifier(strategy='prior')
    if name.startswith('logreg'): return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(C=p['C'],max_iter=1000))
    return HistGradientBoostingClassifier(max_depth=p['max_depth'],learning_rate=0.05,random_state=SEED)
feats={'baseline':ALL,'logreg_all':ALL,'logreg_pre_session':PRE,'gbm_all':ALL}
candidates=[('baseline',{})]+[('logreg_all',{'C':C}) for C in (0.01,0.1,1,10)]+[('logreg_pre_session',{'C':C}) for C in (0.01,0.1,1,10)]+[('gbm_all',{'max_depth':d}) for d in (2,3)]
folds=list(GroupKFold(n_splits=5).split(train,y,g))
res=[];preds={}
for n,p in candidates:
    X=train[feats[n]]; pr=np.zeros(len(X))
    for a,b in folds:
        pr[b]=mk(n,**p).fit(X.iloc[a],y.iloc[a]).predict_proba(X.iloc[b])[:,1]
    preds[(n,str(p))]=pr
    fa=[roc_auc_score(y.iloc[b],pr[b]) for _,b in folds]
    res.append(dict(model=n,params=p,cv_auc=np.mean(fa),auc_sd=np.std(fa),cv_brier=brier_score_loss(y,pr)))
cvres=pd.DataFrame(res).sort_values('cv_auc',ascending=False).reset_index(drop=True); cvres

## 3. Choose model and threshold on CV only

In [ ]:
best=cvres.iloc[0]; name,params=best.model,best.params
pr=preds[(name,str(params))]
ths=np.linspace(0.05,0.95,91)
bal=[balanced_accuracy_score(y,pr>=t) for t in ths]
thr=float(ths[int(np.argmax(bal))])
print('CHOSEN:',name,params,'| CV AUC %.3f'%best.cv_auc,'| threshold %.2f, CV balanced acc %.3f'%(thr,max(bal)))

**Choice, fixed before the test set is touched:** the model, hyperparameters and threshold printed above. Nothing below changes them.

## 4. Refit on all training rows; score the test set once

In [ ]:
final=mk(name,**params).fit(train[feats[name]],y)
yt=test.passed_quiz
p=final.predict_proba(test[feats[name]])[:,1]
base=mk('baseline').fit(train[ALL],y).predict_proba(test[ALL])[:,1]
rng=np.random.default_rng(SEED); ids=test.student_key.unique(); grp=test.groupby('student_key').indices
bs=[]
for _ in range(1000):
    idx=np.concatenate([grp[s] for s in rng.choice(ids,len(ids))])
    if yt.iloc[idx].nunique()>1: bs.append(roc_auc_score(yt.iloc[idx],p[idx]))
print('TEST AUC %.3f (95%% student-bootstrap CI %.3f-%.3f); baseline AUC %.3f'%(roc_auc_score(yt,p),*np.percentile(bs,[2.5,97.5]),roc_auc_score(yt,base)))
print('TEST balanced acc %.3f, accuracy %.3f at threshold %.2f (majority-class accuracy %.3f)'%(balanced_accuracy_score(yt,p>=thr),accuracy_score(yt,p>=thr),thr,1-yt.mean()))

## 5. Interpretation
The headline is the test AUC above, with a student-level bootstrap interval because a student's sessions are correlated. Compare it with the baseline (0.5 by construction) and with the CV table, which shows how much signal comes from the in-session columns versus the pre-session ones. Limitations: no date column, so no temporal check; `prior_score` varies across a student's rows, so its provenance is unclear; the in-session features rest on the timing assumption at the top; `school_code` is treated as numeric.